# 02 项目环境与第一次运行

这一章把第 01 章搭好的环境真正跑起来：确认 Jupyter 使用的是正确的 Python，确认项目根目录、目录结构和样例数据都能被 notebook 识别，然后写出第一个健康检查结果。

完成本章后，你应该能确认四件事：

- Jupyter 可以从 `notebooks/` 中稳定运行本项目的 notebook。
- 当前 kernel 使用的是预期的 Python 环境和依赖包。
- `notebooks/`、`lib/`、`data/sample/`、`outputs/results/` 这些路径可以互相配合。
- 项目自带的 ETF 样例数据可以被读取、检查，并产出一个结果文件。


## 2.1 学习目标与运行顺序

第 01 章解决“环境怎么装”，本章解决“环境是否真的能跑”。只有这一章顺利通过，第 03 章开始的 pandas / numpy 数据处理才有稳定基础。

运行前先确认：

- 已经在 `pyquant-roadmap` 项目中打开 Jupyter，最好从项目根目录或 `notebooks/` 目录启动。
- 已按 `environment.yml` 创建并激活了 Python 3.11 环境。
- Jupyter 当前 notebook 选择了这个环境对应的 kernel。
- `data/sample/` 目录保留了项目随附的 ETF 样例数据。

本章会产出两个检查结果：

| 类型 | 说明 |
|---|---|
| 屏幕输出 | 多张表格会展示项目路径、Jupyter kernel、Python 依赖和样例数据状态。 |
| 文件输出 | 运行成功后会写出 `outputs/results/02_first_run_healthcheck.csv`，作为第一次完整跑通的证明。 |


## 2.2 定位项目根目录

notebook 是学习入口，但代码通常会依赖项目根目录。下面的检测允许你从 `pyquant-roadmap/` 或 `pyquant-roadmap/notebooks/` 启动 Jupyter；只要项目结构完整，后续 `lib` 导入和相对路径都会保持一致。


In [1]:
from __future__ import annotations

import importlib.metadata as metadata
import importlib.util
import json
import os
import platform
import sys
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    """Find pyquant-roadmap from the current notebook working directory."""
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        has_project_files = all(
            (candidate / name).exists()
            for name in ["pyproject.toml", "environment.yml", "notebooks", "lib"]
        )
        if has_project_files:
            return candidate
    raise RuntimeError("没有找到 pyquant-roadmap 项目根目录。请从项目目录或 notebooks/ 目录启动 Jupyter。")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

cwd = Path.cwd().resolve()
try:
    cwd_display = cwd.relative_to(PROJECT_ROOT).as_posix()
except ValueError:
    cwd_display = cwd.name

{
    "current_working_directory": cwd_display,
    "project_root": ".",
    "python_executable": Path(sys.executable).name,
}


{'current_working_directory': 'notebooks',
 'project_root': '.',
 'python_executable': 'python.exe'}

**如何解读**：`project_root` 应该显示为 `.`，含义是当前 notebook 已经定位到 `pyquant-roadmap` 项目根目录。如果这里报错，通常是 Jupyter 在项目外启动，或者项目文件被移动后目录结构不再完整。


## 2.3 确认 Python 环境与 Jupyter kernel

接下来检查核心依赖包和 kernel 信息。量化流程会用到 pandas、numpy、matplotlib、pyarrow、AKShare、bt、quantstats、ta 等包；如果 notebook 连到错误的 Python 环境，这里通常会最先暴露问题。


In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

REQUIRED_PACKAGES = [
    ("pandas", "pandas", "????"),
    ("numpy", "numpy", "????"),
    ("matplotlib", "matplotlib", "??"),
    ("pyarrow", "pyarrow", "?? parquet ??"),
    ("yaml", "PyYAML", "?? yml ??"),
    ("akshare", "akshare", "? 04 ???????"),
    ("bt", "bt", "? 09 ???????"),
    ("quantstats", "quantstats", "? 10 ?????"),
    ("ta", "ta", "? 06 ?????"),
    ("ipykernel", "ipykernel", "Jupyter kernel"),
    ("nbconvert", "nbconvert", "notebook ??????"),
]

package_rows = []
for import_name, dist_name, purpose in REQUIRED_PACKAGES:
    installed = importlib.util.find_spec(import_name) is not None
    try:
        version = metadata.version(dist_name) if installed else None
    except metadata.PackageNotFoundError:
        version = "unknown"
    package_rows.append(
        {
            "import_name": import_name,
            "version": version,
            "purpose": purpose,
            "installed": installed,
        }
    )

package_check = pd.DataFrame(package_rows)
display(package_check)

missing = package_check.loc[~package_check["installed"], "import_name"].tolist()
if missing:
    raise RuntimeError(f"?????????{missing}???? Jupyter ?????????")


In [ ]:
from jupyter_client.kernelspec import KernelSpecManager

NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "02_project_env_first_run.ipynb"
notebook_metadata = json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8")).get("metadata", {})
notebook_kernel = notebook_metadata.get("kernelspec", {})
registered_kernels = KernelSpecManager().find_kernel_specs()

conda_env = os.environ.get("CONDA_DEFAULT_ENV", "")
executable_path = Path(sys.executable).resolve()
executable_display = executable_path.name

kernel_check = pd.DataFrame(
    [
        {
            "check": "Python ??",
            "observed": platform.python_version(),
            "expected": "3.11 ???",
            "ok": sys.version_info >= (3, 11),
        },
        {
            "check": "?????",
            "observed": executable_display,
            "expected": "???????????",
            "ok": True,
        },
        {
            "check": "CONDA_DEFAULT_ENV",
            "observed": conda_env or "???",
            "expected": "????????",
            "ok": True,
        },
        {
            "check": "notebook kernelspec",
            "observed": notebook_kernel.get("name", "???"),
            "expected": "????? notebook ? Python kernel",
            "ok": True,
        },
        {
            "check": "????? kernel ??",
            "observed": len(registered_kernels),
            "expected": "?? 1 ?",
            "ok": len(registered_kernels) >= 1,
        },
    ]
)

display(kernel_check)

if sys.version_info < (3, 11):
    raise RuntimeError("?? Python ???????? Python 3.11 ??????")


**如何解读**：依赖检查表中的 `installed` 应该全部为 `True`，Python 版本至少为 3.11。`CONDA_DEFAULT_ENV` 不一定在所有启动方式下都有值，但 `python_executable`、`notebook kernelspec` 和已安装包能帮助你判断当前 notebook 是否连到了正确环境。


## 2.4 从学习者视角看项目结构

本项目的目录不是让读者先背下来，而是让每个目录服务于后续量化流程：notebook 负责教学步骤，`lib/` 负责复用能力，`data/sample/` 保存缓存数据，`outputs/results/` 保存运行结果。


In [4]:
from lib.paths import (
    CONFIG_DIR,
    DATA_DIR,
    OUTPUT_DIR,
    PROCESSED_DIR,
    RAW_DIR,
    RESULTS_DIR,
    SAMPLE_DIR,
    PROJECT_ROOT as LIB_PROJECT_ROOT,
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

path_rows = [
    ("notebooks", PROJECT_ROOT / "notebooks", "按顺序运行的学习主线"),
    ("lib", PROJECT_ROOT / "lib", "后续章节复用的数据、因子、组合、回测、报告能力"),
    ("configs", CONFIG_DIR, "ETF 池和策略参数等可配置输入"),
    ("data/sample", SAMPLE_DIR, "AKShare 获取后缓存的真实 ETF 样例数据"),
    ("data/raw", RAW_DIR, "保留原始或临时数据的位置"),
    ("data/processed", PROCESSED_DIR, "清洗、对齐后数据的可选落点"),
    ("outputs/results", RESULTS_DIR, "notebook 跑出的图表、表格、信号和报告"),
]

path_check = pd.DataFrame(
    [
        {
            "name": name,
            "relative_path": path.relative_to(PROJECT_ROOT).as_posix(),
            "exists": path.exists(),
            "purpose": purpose,
        }
        for name, path, purpose in path_rows
    ]
)

display(path_check)

if LIB_PROJECT_ROOT.resolve() != PROJECT_ROOT.resolve():
    raise RuntimeError(f"lib.paths.PROJECT_ROOT={LIB_PROJECT_ROOT} 与当前定位的 PROJECT_ROOT={PROJECT_ROOT} 不一致")
if not path_check["exists"].all():
    missing_paths = path_check.loc[~path_check["exists"], "relative_path"].tolist()
    raise FileNotFoundError(f"项目结构缺少这些路径：{missing_paths}")


,name,relative_path,exists,purpose
0,notebooks,notebooks,True,按顺序运行的学习主线
1,lib,lib,True,后续章节复用的数据、因子、组合、回测、报告能力
2,configs,configs,True,ETF 池和策略参数等可配置输入
3,data/sample,data/sample,True,AKShare 获取后缓存的真实 ETF 样例数据
4,data/raw,data/raw,True,保留原始或临时数据的位置
5,data/processed,data/processed,True,清洗、对齐后数据的可选落点
6,outputs/results,outputs/results,True,notebook 跑出的图表、表格、信号和报告


**如何解读**：后续学习主要在 `notebooks/` 中进行。这张表只检查运行项目所需的核心路径；如果某个路径不存在，先修复目录结构，再继续往下跑。


## 2.5 检查缓存数据路径

本章不联网拉取行情，而是使用项目随附的 `data/sample/`。这些文件是第 04 章同类流程通过 AKShare 获取并缓存后的真实 ETF 日线数据；这里先确认缓存文件存在、大小合理，并能被 pyarrow 读取。


In [5]:
import yaml

cache_files = [
    ("assets.parquet", "ETF 资产清单"),
    ("prices.parquet", "标准 OHLCV 日线行情"),
    ("calendar.parquet", "交易日历"),
]

cache_check = pd.DataFrame(
    [
        {
            "file": filename,
            "relative_path": (SAMPLE_DIR / filename).relative_to(PROJECT_ROOT).as_posix(),
            "purpose": purpose,
            "exists": (SAMPLE_DIR / filename).exists(),
            "size_kb": round((SAMPLE_DIR / filename).stat().st_size / 1024, 1) if (SAMPLE_DIR / filename).exists() else 0,
        }
        for filename, purpose in cache_files
    ]
)

config = yaml.safe_load((CONFIG_DIR / "data_sources.yml").read_text(encoding="utf-8"))
configured_symbols = pd.DataFrame(config["akshare_etf_universe"]["symbols"])

display(cache_check)
display(configured_symbols)

if not cache_check["exists"].all():
    missing_cache = cache_check.loc[~cache_check["exists"], "relative_path"].tolist()
    raise FileNotFoundError(f"样例缓存缺失：{missing_cache}。请先确认 data/sample/ 已随项目保留，或在第 04 章重新获取。")


,file,relative_path,purpose,exists,size_kb
0,assets.parquet,data/sample/assets.parquet,ETF 资产清单,True,2.7
1,prices.parquet,data/sample/prices.parquet,标准 OHLCV 日线行情,True,104.4
2,calendar.parquet,data/sample/calendar.parquet,交易日历,True,7.8


,code,name
0,510300,沪深300ETF
1,510500,中证500ETF
2,159915,创业板ETF
3,512100,中证1000ETF


**如何解读**：`configured_symbols` 是项目默认 ETF 池。缓存文件存在后，后续章节可以直接读取本地 parquet，避免每章都请求网络，也让学习结果更容易复现。


## 2.6 第一次读取真实 ETF 样例数据

现在从 `lib.data` 读取缓存。这里不是用随机数造演示表，而是读取已经标准化为 `date/code/open/high/low/close/volume/amount` 的真实 ETF 数据，为后续收益率、因子和回测章节做准备。


In [6]:
from lib.data import load_sample_assets, load_sample_calendar, load_sample_prices

assets = load_sample_assets()
prices = load_sample_prices()
calendar = load_sample_calendar()

prices = prices.copy()
prices["date"] = pd.to_datetime(prices["date"])
calendar = calendar.copy()
calendar["date"] = pd.to_datetime(calendar["date"])

display(assets)
display(prices.head())


,code,name,asset_type,list_date
0,510300,沪深300ETF,ETF,2000-01-01
1,510500,中证500ETF,ETF,2000-01-01
2,159915,创业板ETF,ETF,2000-01-01
3,512100,中证1000ETF,ETF,2000-01-01


,date,code,open,high,low,close,volume,amount
0,2021-01-04,159915,2.864,2.990,2.861,2.976,2472764,7.278150e+08
1,2021-01-04,510300,4.789,4.875,4.769,4.843,5067056,2.697193e+09
2,2021-01-04,510500,5.926,6.046,5.899,6.014,3035862,2.166750e+09
3,2021-01-04,512100,2.493,2.545,2.488,2.537,1121520,1.064703e+08
4,2021-01-05,159915,2.940,2.997,2.925,2.988,2470004,7.346481e+08


In [7]:
coverage = (
    prices.groupby("code")
    .agg(
        start=("date", "min"),
        end=("date", "max"),
        rows=("date", "size"),
        first_close=("close", "first"),
        last_close=("close", "last"),
    )
    .reset_index()
    .merge(assets[["code", "name"]], on="code", how="left")
    [["code", "name", "start", "end", "rows", "first_close", "last_close"]]
)

display(coverage)
calendar_summary = pd.DataFrame(
    [
        ("start", calendar["date"].min().date().isoformat()),
        ("end", calendar["date"].max().date().isoformat()),
        ("open_days", int(calendar["is_open"].sum())),
    ],
    columns=["item", "value"],
)
display(calendar_summary)


,code,name,start,end,rows,first_close,last_close
0,159915,创业板ETF,2021-01-04,2023-12-29,725,2.976,1.842
1,510300,沪深300ETF,2021-01-04,2023-12-29,725,4.843,3.219
2,510500,中证500ETF,2021-01-04,2023-12-29,725,6.014,5.279
3,512100,中证1000ETF,2021-01-04,2023-12-29,725,2.537,2.296


,item,value
0,start,2021-01-04
1,end,2023-12-29
2,open_days,725


**如何解读**：每只 ETF 都应该有相近的起止日期和行数。第 03 章会把这张长表转成矩阵，并计算收益率、动量、波动率等常用字段。


## 2.7 做几个小质量检查

环境跑通不只是不报错，还要确认读取到的数据符合后续章节的最小假设：字段完整、日期可排序、每个价格代码都在资产清单里，并且没有重复的 `date + code`。


In [8]:
required_price_columns = {"date", "code", "open", "high", "low", "close", "volume", "amount"}
duplicate_rows = prices.duplicated(["date", "code"]).sum()
unknown_codes = sorted(set(prices["code"]) - set(assets["code"]))

data_checks = pd.DataFrame(
    [
        {
            "check": "价格字段完整",
            "observed": sorted(prices.columns.tolist()),
            "ok": required_price_columns.issubset(prices.columns),
        },
        {
            "check": "价格数据非空",
            "observed": len(prices),
            "ok": len(prices) > 0,
        },
        {
            "check": "date 已转成 datetime",
            "observed": str(prices["date"].dtype),
            "ok": pd.api.types.is_datetime64_any_dtype(prices["date"]),
        },
        {
            "check": "close 没有缺失值",
            "observed": int(prices["close"].isna().sum()),
            "ok": prices["close"].isna().sum() == 0,
        },
        {
            "check": "没有重复 date/code",
            "observed": int(duplicate_rows),
            "ok": duplicate_rows == 0,
        },
        {
            "check": "价格代码都在资产清单中",
            "observed": unknown_codes or "全部匹配",
            "ok": len(unknown_codes) == 0,
        },
        {
            "check": "交易日历覆盖价格区间",
            "observed": f"{calendar['date'].min().date()} 至 {calendar['date'].max().date()}",
            "ok": calendar["date"].min() <= prices["date"].min() and calendar["date"].max() >= prices["date"].max(),
        },
    ]
)

display(data_checks)

failed_data_checks = data_checks.loc[~data_checks["ok"], "check"].tolist()
if failed_data_checks:
    raise RuntimeError(f"样例数据检查未通过：{failed_data_checks}")


,check,observed,ok
0,价格字段完整,"[amount, close, code, date, high, low, open, v...",True
1,价格数据非空,2900,True
2,date 已转成 datetime,datetime64[ns],True
3,close 没有缺失值,0,True
4,没有重复 date/code,0,True
5,价格代码都在资产清单中,全部匹配,True
6,交易日历覆盖价格区间,2021-01-04 至 2023-12-29,True


## 2.8 写出第一个很小的结果

真正的策略报告要到后面章节才会生成。本章只写一个健康检查 CSV，证明 notebook 可以把结果落到 `outputs/results/`，同时验证路径规则和文件编码都正常。


In [ ]:
healthcheck_path = RESULTS_DIR / "02_first_run_healthcheck.csv"

healthcheck = pd.DataFrame(
    [
        ("project_root", "."),
        ("python", platform.python_version()),
        ("python_executable", Path(sys.executable).name),
        ("conda_env", conda_env or "???"),
        ("kernel_name", notebook_kernel.get("name", "???")),
        ("asset_count", len(assets)),
        ("price_rows", len(prices)),
        ("first_price_date", prices["date"].min().date().isoformat()),
        ("last_price_date", prices["date"].max().date().isoformat()),
    ],
    columns=["item", "value"],
)

healthcheck.to_csv(healthcheck_path, index=False, encoding="utf-8")

print(f"????{healthcheck_path.relative_to(PROJECT_ROOT).as_posix()}")
display(healthcheck)


In [ ]:
pd.read_csv(healthcheck_path).head(20)


**如何解读**：这个 CSV 不是策略结果，只是第一次运行的证明。后续章节会在同一个目录下继续生成因子、净值、绩效、目标权重和订单建议等文件。


## 2.9 小练习：筛出异常路径

练习：从 `path_check` 表里筛出不存在的路径。正常情况下结果应该是空表；如果不是空表，先修复目录结构，再继续后面的 notebook。


In [11]:
# 练习区：补全筛选条件。
# path_check.loc[...]


参考答案如下。先自己运行上一格练习区，再用这里的结果对照。


In [12]:
missing_paths = path_check.loc[~path_check["exists"], ["name", "relative_path", "purpose"]]
missing_paths


,name,relative_path,purpose


## 2.10 常见问题排查

| 现象 | 常见原因 | 处理方式 |
|---|---|---|
| `ModuleNotFoundError: No module named 'lib'` | Jupyter 没有从项目目录启动，或根目录没有被加入 `sys.path`。 | 回到第 2.2 节，确认 `PROJECT_ROOT` 指向 `pyquant-roadmap`。 |
| 依赖检查里出现 `installed=False` | 当前 notebook 连到的不是安装过依赖的环境。 | 在 Jupyter 里切换到正确 kernel，必要时重启 Jupyter 后重新打开 notebook。 |
| `data/sample/*.parquet` 缺失 | 样例缓存没有随项目保留，或文件被移动。 | 先恢复 `data/sample/`；也可以在第 04 章按 AKShare 流程重新获取缓存。 |
| 写结果时报错 | `outputs/results/` 路径不存在或没有写入权限。 | 确认项目目录可写，并让本章创建或恢复 `outputs/results/`。 |

排查时优先看最早报错的单元。一般先解决路径和 kernel 问题，再继续排查数据文件；这样可以避免后面的错误被前面的环境问题放大。


## 2.11 交接到第 03 章

到这里，环境、kernel、导入路径、数据缓存和结果输出都已经验证过。第 03 章会接着使用刚读出的 `prices` 长表，练习 pandas / numpy 在量化里最常用的四件事：按资产做时间序列计算、按日期做横截面比较、长表转矩阵、用 `shift` 避免未来函数。
